# Running Back OQA Analysis: Carry-Adjusted Performance vs. Opponent Expectations

## Hypothesis

Raw rushing yards mislead. Eric Dickerson's 1,659-yard season looks better than Roger Craig's 1,502 yards — but
Dickerson faced softer run defenses. The **carry-adjusted OQA metric** levels the playing field:

```
expected_rush_yds  = player_rush_att × defense_LOO_avg_rush_ypc_allowed
delta_rush_yds     = actual_rush_yds − expected_rush_yds
```

The defense LOO average excludes the current game (leave-one-out) to prevent circularity. Positive delta means
the back gained more yards per carry than the defense typically allows. Negative means held below average.

**Key distinction from QB analysis**: We are not measuring wins — rushing efficiency is
more about individual brilliance and offensive line support than team record. The story here is
*who was actually the best runner given who they faced?*

**Data**: 1960–2024 boxscores via PFR. Carry-adjusted deltas computed in `scripts/etl_player_game_offense.py`.

In [1]:
import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

OQA_DIR   = '/Users/devos/github/football_analytics/data_output/'
GAME_DIR  = '/Users/devos/github/football_analytics/data_output/'
DEF_DIR   = '/Users/devos/github/football_analytics/data_output/'

## Load Season OQA Data

In [2]:
frames = []
for path in sorted(glob.glob(f'{OQA_DIR}player_oqa_season_*.csv')):
    year = int(path.split('_')[-1].replace('.csv', ''))
    if year < 1960:
        continue
    df = pd.read_csv(path, low_memory=False)
    df['season'] = year
    frames.append(df)

oqa_raw = pd.concat(frames, ignore_index=True)
for col in ['total_rush_yds', 'total_rush_td', 'total_rec_yds', 'total_rec',
            'delta_rush_yds_total', 'delta_rush_yds_pg',
            'delta_rush_td_total', 'delta_rush_td_pg',
            'delta_rec_yds_total', 'delta_rec_yds_pg', 'games']:
    oqa_raw[col] = pd.to_numeric(oqa_raw[col], errors='coerce')

rb_all = oqa_raw[oqa_raw['position_group'] == 'RB'].copy()
print(f'All RB season rows: {len(rb_all):,}  ({rb_all.season.min()}–{rb_all.season.max()})')

# Featured back filter: ≥100 rush attempts proxy → ≥400 rush yards AND ≥8 games played
# (We don't have attempt counts in the season summary, so use yards as proxy)
rb_featured = rb_all[
    (rb_all['total_rush_yds'] >= 400) &
    (rb_all['games'] >= 8)
].copy()
print(f'Featured back seasons (≥400 rush yds, ≥8 games): {len(rb_featured):,}')

All RB season rows: 9,057  (1960–2025)
Featured back seasons (≥400 rush yds, ≥8 games): 2,481


## Load Game-Level Data (for individual game highlights)

In [3]:
game_frames = []
def_frames  = []

for path in sorted(glob.glob(f'{GAME_DIR}player_game_offense_*.csv')):
    year = int(path.split('_')[-1].replace('.csv', ''))
    if year < 1960:
        continue
    df = pd.read_csv(path, low_memory=False)
    df['season'] = year
    game_frames.append(df[df['position_group'] == 'RB'])

for path in sorted(glob.glob(f'{DEF_DIR}defense_loo_avg_*.csv')):
    year = int(path.split('_')[-1].replace('.csv', ''))
    if year < 1960:
        continue
    df = pd.read_csv(path, low_memory=False)
    df['season'] = year
    def_frames.append(df)

games_rb  = pd.concat(game_frames, ignore_index=True)
def_loo   = pd.concat(def_frames,  ignore_index=True)

for col in ['rush_att', 'rush_yds', 'rush_td', 'rec', 'rec_yds', 'rec_td']:
    games_rb[col] = pd.to_numeric(games_rb[col], errors='coerce').fillna(0)

# Merge defense LOO stats: must join on both game_id AND the defending team
# (each game has two rows in def_loo — one per team — so joining on game_id alone duplicates rows)
games_rb = games_rb.merge(
    def_loo[['game_id', 'defending_team', 'def_avg_rush_ypc', 'def_avg_rec_yds_rb']],
    left_on=['game_id', 'defense_team'],
    right_on=['game_id', 'defending_team'],
    how='left'
).drop(columns=['defending_team'])

# Carry-adjusted delta per game
games_rb['expected_rush_yds'] = games_rb['rush_att'] * games_rb['def_avg_rush_ypc']
games_rb['delta_rush_yds']    = games_rb['rush_yds'] - games_rb['expected_rush_yds']

print(f'RB game rows: {len(games_rb):,}  (with ≥1 rush attempt: {(games_rb.rush_att >= 1).sum():,})')

RB game rows: 81,168  (with ≥1 rush attempt: 74,198)


## Validate Test Cases

From the framework doc — these are our ground-truth checks before any analysis.

In [4]:
def show_rb_season(player_name, year, top_n_games=3):
    """Print season summary + top games for an RB."""
    row = rb_all[
        (rb_all['season'] == year) &
        (rb_all['player_name'].str.contains(player_name, na=False))
    ]
    if row.empty:
        print(f'  {player_name} {year}: NOT FOUND'); return
    r = row.iloc[0]
    print(f'\n{r.player_name} — {r.team_abbrev} {year}')
    print(f'  Rush yards: {int(r.total_rush_yds):,}   Rush TDs: {r.total_rush_td}')
    print(f'  Δ rush total (carry-adj): {r.delta_rush_yds_total:+.1f}   per game: {r.delta_rush_yds_pg:+.1f}')
    print(f'  Rec yards: {int(r.total_rec_yds):,}   Δ rec total: {r.delta_rec_yds_total:+.1f}')

    # Top games from game-level data
    g = games_rb[
        (games_rb['season'] == year) &
        (games_rb['player_name'].str.contains(player_name, na=False)) &
        (games_rb['rush_att'] >= 5)
    ].copy()
    if not g.empty:
        g['_lbl'] = g['game_id'].str[-3:].str.upper()  # home team abbrev
        top = g.nlargest(top_n_games, 'delta_rush_yds')[['_lbl','rush_att','rush_yds','expected_rush_yds','delta_rush_yds','def_avg_rush_ypc']]
        print(f'  Top {top_n_games} games by delta:')
        for _, gm in top.iterrows():
            print(f'    vs {gm._lbl}: {int(gm.rush_att)} att, {int(gm.rush_yds)} yds '
                  f'(exp {gm.expected_rush_yds:.1f} @ {gm.def_avg_rush_ypc:.2f} ypc) '
                  f'→ Δ {gm.delta_rush_yds:+.1f}')

# Known values from framework doc:
# Barry Sanders 1997: +712.4 total, +44.52/game
# Ickey Woods 1988:   +276.1 total, +18.41/game
# Eric Dickerson 1988: +105.0 total, +6.56/game
# Roger Craig 1988:   +240.7 total, +15.04/game

for player, year in [('Barry Sanders', 1997), ('Ickey Woods', 1988),
                     ('Eric Dickerson', 1988), ('Roger Craig', 1988)]:
    show_rb_season(player, year)


Barry Sanders — DET 1997
  Rush yards: 2,053   Rush TDs: 11.0
  Δ rush total (carry-adj): +712.4   per game: +44.5
  Rec yards: 305   Δ rec total: -374.8
  Top 3 games by delta:
    vs TAM: 24 att, 215 yds (exp 84.2 @ 3.51 ypc) → Δ +130.8
    vs DET: 24 att, 216 yds (exp 106.6 @ 4.44 ypc) → Δ +109.4
    vs DET: 23 att, 184 yds (exp 88.8 @ 3.86 ypc) → Δ +95.2

Ickey Woods — CIN 1988
  Rush yards: 1,066   Rush TDs: 15.0
  Δ rush total (carry-adj): +276.1   per game: +18.4
  Rec yards: 199   Δ rec total: -583.6
  Top 3 games by delta:
    vs CIN: 10 att, 110 yds (exp 34.2 @ 3.42 ypc) → Δ +75.8
    vs CIN: 19 att, 141 yds (exp 75.4 @ 3.97 ypc) → Δ +65.6
    vs CIN: 18 att, 115 yds (exp 70.0 @ 3.89 ypc) → Δ +45.0

Eric Dickerson — IND 1988
  Rush yards: 1,659   Rush TDs: 14.0
  Δ rush total (carry-adj): +105.0   per game: +6.6
  Rec yards: 377   Δ rec total: -474.3
  Top 3 games by delta:
    vs CLT: 21 att, 159 yds (exp 95.1 @ 4.53 ypc) → Δ +63.9
    vs SDG: 30 att, 169 yds (exp 121.2 @ 4

---
# Chart 1 — Top 50 Single Seasons: Carry-Adjusted Rush Yards Above Expected

Every featured back season ranked by **total carry-adjusted yards above expected**.
Color = era. Hover for full context.

In [5]:
def season_games(year):
    if year <= 1960: return 12
    if year <= 1977: return 14
    if year <= 2020: return 16
    return 17

rb_featured['season_len'] = rb_featured['season'].apply(season_games)

def era_label(year):
    if year < 1970: return '1960s'
    if year < 1980: return '1970s'
    if year < 1990: return '1980s'
    if year < 2000: return '1990s'
    if year < 2010: return '2000s'
    if year < 2020: return '2010s'
    return '2020s'

rb_featured['era'] = rb_featured['season'].apply(era_label)

top50 = rb_featured.nlargest(50, 'delta_rush_yds_total').reset_index(drop=True)
top50['label'] = top50['player_name'].str.split().str[-1] + "'" + top50['season'].astype(str).str[-2:]
top50['hover'] = (
    '<b>' + top50['player_name'] + '</b> — ' + top50['team_abbrev'] + ' ' + top50['season'].astype(str) +
    '<br>Rush yards: ' + top50['total_rush_yds'].astype(int).astype(str) +
    '  TDs: ' + top50['total_rush_td'].fillna(0).astype(int).astype(str) +
    '<br>Δ total (carry-adj): <b>' + top50['delta_rush_yds_total'].round(1).astype(str) + '</b>' +
    '<br>Δ per game: ' + top50['delta_rush_yds_pg'].round(1).astype(str) +
    '<br>Games: ' + top50['games'].astype(int).astype(str)
)

era_colors = {
    '1960s': '#6a3d9a', '1970s': '#1f78b4', '1980s': '#33a02c',
    '1990s': '#ff7f00', '2000s': '#e31a1c', '2010s': '#a6cee3', '2020s': '#b2df8a'
}
bar_colors = [era_colors[e] for e in top50['era']]

fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=top50['label'],
    y=top50['delta_rush_yds_total'],
    marker_color=bar_colors,
    hovertext=top50['hover'],
    hoverinfo='text',
    name='Δ Rush Yds (carry-adj)',
))

# Era legend traces (invisible, for legend only)
for era, color in era_colors.items():
    if era in top50['era'].values:
        fig1.add_trace(go.Bar(x=[None], y=[None], marker_color=color, name=era, showlegend=True))

fig1.update_layout(
    title=dict(text='Top 50 Single-Season Carry-Adjusted Rush Yards Above Expected (1960–2024)<br>'
                    '<sup>Δ = actual rush yds − (rush att × defense\'s avg YPC allowed, leave-one-out)</sup>',
               x=0.5),
    xaxis=dict(title='', tickangle=50, tickfont=dict(size=9)),
    yaxis=dict(title='Δ Rush Yards (Carry-Adjusted)'),
    height=520,
    hovermode='closest',
    legend=dict(title='Era', orientation='v', x=1.01),
    bargap=0.15,
)
fig1.show()

print('Top 10 seasons:')
print(top50[['player_name','team_abbrev','season','total_rush_yds','delta_rush_yds_total','delta_rush_yds_pg']]
      .head(10).rename(columns={'player_name':'Player','team_abbrev':'Tm','season':'Year',
                                 'total_rush_yds':'Rush Yds','delta_rush_yds_total':'Δ Total',
                                 'delta_rush_yds_pg':'Δ/game'}).round(1).to_string(index=False))

Top 10 seasons:
         Player  Tm  Year  Rush Yds  Δ Total  Δ/game
  Barry Sanders DET  1997    2053.0    712.4    44.5
      Jim Brown CLE  1963    1863.0    700.2    50.0
  Barry Sanders DET  1994    1883.0    653.8    40.9
Adrian Peterson MIN  2012    2097.0    638.5    39.9
   O.J. Simpson BUF  1973    2003.0    603.5    43.1
 Eric Dickerson RAM  1984    2105.0    576.2    36.0
  Walter Payton CHI  1977    1852.0    550.4    39.3
  Terrell Davis DEN  1998    2008.0    506.4    31.6
  Earl Campbell HOU  1980    1934.0    500.8    33.4
 Jamaal Charles KAN  2010    1467.0    484.3    30.3


---
# Chart 2 — Raw Yards vs. Carry-Adjusted Yards: Who Was Over/Underrated?

Each dot = one featured back season (≥400 rush yds, ≥8 games).
- **X-axis**: actual rush yards
- **Y-axis**: carry-adjusted delta total
- **Quadrants**: top-left = great efficiency on few yards; bottom-right = volume back facing soft defenses
- Named GOAT-tier backs highlighted

In [6]:
NAMED_RBS = {
    'Jim Brown':          '#8B0000',
    'Gale Sayers':        '#9400D3',
    'O.J. Simpson':       '#FF4500',
    'Earl Campbell':      '#8B4513',
    'Walter Payton':      '#006400',
    'Tony Dorsett':       '#00008B',
    'Eric Dickerson':     '#FF8C00',
    'Bo Jackson':         '#2E8B57',
    'Barry Sanders':      '#1565C0',
    'Emmitt Smith':       '#C0392B',
    'Thurman Thomas':     '#6D28D9',
    'Marshall Faulk':     '#0E6655',
    'Terrell Davis':      '#922B21',
    'LaDainian Tomlinson':'#1A5276',
    'Adrian Peterson':    '#148F77',
    'Jamaal Charles':     '#7D6608',
    'Le\'Veon Bell':      '#4A235A',
    'Ezekiel Elliott':    '#1E3A5F',
    'Derrick Henry':      '#6E2F1A',
    'Christian McCaffrey':'#0D47A1',
}

fig2 = go.Figure()

# Background: all featured seasons
bg = rb_featured.copy()
hover_bg = (
    bg['player_name'] + ' — ' + bg['team_abbrev'] + ' ' + bg['season'].astype(str) +
    '<br>Rush yds: ' + bg['total_rush_yds'].astype(int).astype(str) +
    '  Δ total: ' + bg['delta_rush_yds_total'].round(1).astype(str)
)
fig2.add_trace(go.Scatter(
    x=bg['total_rush_yds'],
    y=bg['delta_rush_yds_total'],
    mode='markers',
    marker=dict(color='rgba(180,180,180,0.2)', size=4),
    name='All featured backs',
    hovertext=hover_bg, hoverinfo='text',
))

# Named backs
for rb_name, color in NAMED_RBS.items():
    sub = rb_featured[rb_featured['player_name'].str.contains(rb_name.split()[0], na=False) &
                      rb_featured['player_name'].str.contains(rb_name.split()[-1], na=False)]
    if sub.empty:
        continue
    hover = (
        '<b>' + sub['player_name'] + '</b> — ' + sub['team_abbrev'] + ' ' + sub['season'].astype(str) +
        '<br>Rush yds: ' + sub['total_rush_yds'].astype(int).astype(str) +
        '<br>Δ total (carry-adj): ' + sub['delta_rush_yds_total'].round(1).astype(str) +
        '<br>Δ/game: ' + sub['delta_rush_yds_pg'].round(1).astype(str)
    )
    fig2.add_trace(go.Scatter(
        x=sub['total_rush_yds'],
        y=sub['delta_rush_yds_total'],
        mode='markers',
        marker=dict(color=color, size=9, opacity=0.85,
                    line=dict(width=0.8, color='white')),
        name=rb_name,
        hovertext=hover, hoverinfo='text',
    ))

fig2.add_hline(y=0, line_color='black', line_width=1.5)
fig2.add_vline(x=1000, line_dash='dot', line_color='gray', line_width=1,
               annotation_text='1,000 rush yds', annotation_position='top')
fig2.add_vline(x=1500, line_dash='dot', line_color='gray', line_width=1,
               annotation_text='1,500 rush yds', annotation_position='top')

# Quadrant labels
for x, y, txt in [
    (500,  700, 'Elite efficiency,<br>low volume'),
    (1800, 700, 'Elite volume +<br>quality'),
    (1800, -500, 'High volume,<br>soft defenses'),
    (500,  -400, 'Low volume,<br>poor efficiency'),
]:
    fig2.add_annotation(x=x, y=y, text=txt, showarrow=False,
                        font=dict(size=9, color='gray'))

fig2.update_layout(
    title=dict(text='Raw Rush Yards vs. Carry-Adjusted Delta — All Featured Back Seasons 1960–2024<br>'
                    '<sup>Dots above zero line beat opponent expectations; below = held below average YPC.</sup>',
               x=0.5),
    xaxis=dict(title='Actual Rush Yards'),
    yaxis=dict(title='Δ Rush Yards (Carry-Adjusted Total)'),
    height=600,
    hovermode='closest',
    legend=dict(orientation='v', x=1.01, y=1,
                bgcolor='rgba(255,255,255,0.85)',
                bordercolor='lightgray', borderwidth=1),
)
fig2.show()

---
# Chart 3 — Career Rankings: Avg Δ/Game for All-Time Greats

**Method**: For each RB with ≥4 featured back seasons (≥400 rush yds, ≥8 games), compute:
- Average carry-adjusted rush delta **per game** across their career
- Average **total** delta per season
- Career totals

Δ/game is the efficiency measure — it controls for game counts and era schedule length.
Total Δ rewards sustained dominance.

In [7]:
rb_career = (
    rb_featured
    .groupby('player_name').agg(
        seasons           = ('season', 'count'),
        first_year        = ('season', 'min'),
        last_year         = ('season', 'max'),
        avg_rush_yds      = ('total_rush_yds', 'mean'),
        total_rush_yds    = ('total_rush_yds', 'sum'),
        avg_delta_pg      = ('delta_rush_yds_pg', 'mean'),
        avg_delta_total   = ('delta_rush_yds_total', 'mean'),
        career_delta      = ('delta_rush_yds_total', 'sum'),
        avg_rush_td       = ('total_rush_td', 'mean'),
        avg_rec_yds       = ('total_rec_yds', 'mean'),
        avg_delta_rec_pg  = ('delta_rec_yds_pg', 'mean'),
    ).reset_index()
)

# Require ≥4 qualifying seasons
rb_career_q = rb_career[rb_career['seasons'] >= 4].sort_values('avg_delta_pg', ascending=False).reset_index(drop=True)
print(f'RBs with 4+ featured seasons: {len(rb_career_q)}')
print()
print('Top 20 by avg Δ/game:')
print(rb_career_q.head(20)[['player_name','seasons','first_year','last_year',
                              'avg_rush_yds','avg_delta_pg','avg_delta_total','career_delta']]
      .rename(columns={'player_name':'Player','seasons':'Seasons',
                       'first_year':'From','last_year':'To',
                       'avg_rush_yds':'Avg Rush Yds','avg_delta_pg':'Avg Δ/game',
                       'avg_delta_total':'Avg Δ/season','career_delta':'Career Δ'})
      .round(1).to_string(index=False))

RBs with 4+ featured seasons: 281

Top 20 by avg Δ/game:
          Player  Seasons  From   To  Avg Rush Yds  Avg Δ/game  Avg Δ/season  Career Δ
       Jim Brown        6  1960 1965        1446.3        26.3         365.6    2193.5
   Barry Sanders       10  1989 1998        1526.9        21.1         326.4    3263.8
  Jamaal Charles        5  2009 2014        1283.2        20.1         311.8    1559.1
     Gale Sayers        5  1965 1969         973.2        19.8         238.7    1193.4
   Terrell Davis        5  1995 2001        1422.8        16.7         253.8    1269.2
  Mercury Morris        4  1970 1975         809.5        14.7         180.0     720.0
   Priest Holmes        4  2001 2004        1370.5        14.3         192.2     768.8
      Nick Chubb        5  2018 2022        1248.4        14.2         206.1    1030.7
    Robert Smith        6  1995 2000        1052.2        14.0         186.0    1116.1
     Greg Pruitt        5  1974 1978         930.6        12.9         17

In [8]:
# ── Bar chart: all qualified RBs sorted by avg_delta_pg ──────────────────────

# Highlight named RBs
def is_named(name):
    for nm in NAMED_RBS:
        parts = nm.split()
        if parts[0] in name and parts[-1] in name:
            return True
    return False

rb_career_q['is_named'] = rb_career_q['player_name'].apply(is_named)
rb_career_q['bar_color'] = rb_career_q.apply(
    lambda r: ('#1a9850' if r['avg_delta_pg'] >= 0 else '#d73027')
    if not r['is_named'] else '#2471A3', axis=1
)

hover_career = (
    '<b>' + rb_career_q['player_name'] + '</b>' +
    '<br>Career: ' + rb_career_q['first_year'].astype(str) + '–' + rb_career_q['last_year'].astype(str) +
    '  (' + rb_career_q['seasons'].astype(str) + ' qualified seasons)' +
    '<br>Avg rush yds/season: ' + rb_career_q['avg_rush_yds'].round(0).astype(int).astype(str) +
    '<br>Avg Δ/game: <b>' + rb_career_q['avg_delta_pg'].round(2).astype(str) + '</b>' +
    '<br>Avg Δ/season: ' + rb_career_q['avg_delta_total'].round(1).astype(str) +
    '<br>Career total Δ: ' + rb_career_q['career_delta'].round(1).astype(str) +
    '<br>Avg rush TDs: ' + rb_career_q['avg_rush_td'].round(1).astype(str) +
    '<extra></extra>'
)

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=rb_career_q['player_name'],
    y=rb_career_q['avg_delta_pg'],
    marker_color=rb_career_q['bar_color'],
    hovertemplate=hover_career,
    name='Avg Δ/game',
))
fig3.add_hline(y=0, line_color='black', line_width=1.5)

# Mark named backs
for _, row in rb_career_q[rb_career_q['is_named']].iterrows():
    fig3.add_vline(x=row['player_name'], line_dash='dot',
                   line_color='rgba(0,0,0,0.25)', line_width=1)

fig3.update_layout(
    title=dict(text='Career Avg Δ Rush Yards/Game — All RBs with 4+ Featured Seasons<br>'
                    '<sup>Blue = named GOAT-tier RBs. Green/red = all others. Sorted by Avg Δ/game.</sup>',
               x=0.5),
    xaxis=dict(title='', tickangle=55, tickfont=dict(size=8)),
    yaxis=dict(title='Avg Δ Rush Yards per Game (Carry-Adjusted)'),
    height=520,
    hovermode='closest',
    bargap=0.12,
)
fig3.show()

---
# Chart 4 — Best Individual Games Ever (Carry-Adjusted)

The single-game carry-adjusted delta rewards outlier performances against quality defenses.
Barry Sanders' 215-yard game vs. Tampa Bay 1997 is the benchmark: 24 carries, defense
allows 3.51 YPC → expected 84.2 yards → **+130.8 above expected**.

Minimum: 10 rush attempts (avoids garbage time flukes).

In [9]:
top_games = games_rb[
    (games_rb['rush_att'] >= 10) &
    (games_rb['delta_rush_yds'].notna())
].nlargest(100, 'delta_rush_yds').reset_index(drop=True)

top_games['label'] = (
    top_games['player_name'].str.split().str[-1] +
    "'" + top_games['season'].astype(str).str[-2:]
)

def era_color_game(year):
    era_colors = {
        '1960s': '#6a3d9a', '1970s': '#1f78b4', '1980s': '#33a02c',
        '1990s': '#ff7f00', '2000s': '#e31a1c', '2010s': '#a6cee3', '2020s': '#b2df8a'
    }
    return era_colors[era_label(year)]

top_games['color'] = top_games['season'].apply(era_color_game)

top_games['hover_game'] = (
    '<b>' + top_games['player_name'] + '</b> — ' + top_games['team_abbrev'] + ' ' + top_games['season'].astype(str) +
    '<br>Game: ' + top_games['game_id'] +
    '<br>' + top_games['rush_att'].astype(int).astype(str) + ' att, ' +
    top_games['rush_yds'].astype(int).astype(str) + ' yds' +
    '<br>Def YPC allowed: ' + top_games['def_avg_rush_ypc'].round(2).astype(str) +
    '  → expected: ' + top_games['expected_rush_yds'].round(1).astype(str) + ' yds' +
    '<br>Δ: <b>+' + top_games['delta_rush_yds'].round(1).astype(str) + '</b>'
)

fig4 = go.Figure()
fig4.add_trace(go.Bar(
    x=top_games['label'],
    y=top_games['delta_rush_yds'],
    marker_color=top_games['color'],
    hovertext=top_games['hover_game'],
    hoverinfo='text',
    name='Single-game Δ',
))

for era, color in {
    '1960s': '#6a3d9a', '1970s': '#1f78b4', '1980s': '#33a02c',
    '1990s': '#ff7f00', '2000s': '#e31a1c', '2010s': '#a6cee3', '2020s': '#b2df8a'
}.items():
    if era in top_games['season'].apply(era_label).values:
        fig4.add_trace(go.Bar(x=[None], y=[None], marker_color=color, name=era, showlegend=True))

fig4.update_layout(
    title=dict(text='Top 100 Individual Games: Carry-Adjusted Rush Yards Above Expected (≥10 att)<br>'
                    '<sup>Δ = actual yds − (att × defense LOO YPC). Min 10 rush attempts.</sup>',
               x=0.5),
    xaxis=dict(title='', tickangle=55, tickfont=dict(size=8)),
    yaxis=dict(title='Δ Rush Yards (Single Game, Carry-Adjusted)'),
    height=520,
    hovermode='closest',
    legend=dict(title='Era', orientation='v', x=1.01),
    bargap=0.12,
)
fig4.show()

print('Top 15 single games:')
print(top_games[['player_name','team_abbrev','season','game_id','rush_att','rush_yds',
                  'expected_rush_yds','delta_rush_yds']]
      .head(15).rename(columns={'player_name':'Player','team_abbrev':'Tm','season':'Year',
                                 'game_id':'Game','rush_att':'Att','rush_yds':'Yds',
                                 'expected_rush_yds':'Exp Yds','delta_rush_yds':'Δ'})
      .round(1).to_string(index=False))

Top 15 single games:
         Player  Tm  Year         Game  Att  Yds  Exp Yds     Δ
   Corey Dillon CIN  2000 200010220cin   22  278     85.4 192.6
Adrian Peterson MIN  2007 200711040min   30  296    107.4 188.6
   O.J. Simpson BUF  1976 197611250det   29  273    101.5 171.5
 Isaiah Crowell NYJ  2018 201810070nyj   15  219     61.0 158.0
 Jamaal Charles KAN  2009 201001030den   25  259    103.0 156.0
   Derrick Ward NYG  2008 200812210nyg   15  215     61.8 153.2
    Doug Martin TAM  2012 201211040rai   25  251     98.2 152.8
     Frank Gore SFO  2009 200909200sfo   16  207     61.0 146.0
      Jim Brown CLE  1963 196309220dal   20  232     86.2 145.8
     Bo Jackson RAI  1987 198711300sea   18  221     77.2 143.8
   Jahmyr Gibbs DET  2025 202511230det   15  219     75.3 143.7
Adrian Peterson MIN  2007 200710140chi   20  224     80.6 143.4
 Saquon Barkley PHI  2024 202411240ram   26  255    114.1 140.9
 DeMarco Murray DAL  2011 201110230dal   25  253    112.2 140.8
    Ahman Green GNB

---
# Chart 5 — Era Comparison: How Has RB Efficiency Changed Over Time?

The game has changed. Modern defenses are faster and more exotic; modern offenses spread the field.
Does the average featured back outperform expectations more or less than in previous eras?

Each year: median and mean carry-adjusted Δ/game for all featured backs that season.

In [10]:
era_trend = (
    rb_featured
    .groupby('season')
    .agg(
        n_backs    = ('player_name', 'count'),
        mean_delta = ('delta_rush_yds_pg', 'mean'),
        med_delta  = ('delta_rush_yds_pg', 'median'),
        p25_delta  = ('delta_rush_yds_pg', lambda x: x.quantile(0.25)),
        p75_delta  = ('delta_rush_yds_pg', lambda x: x.quantile(0.75)),
        mean_yds   = ('total_rush_yds', 'mean'),
    ).reset_index()
)

fig5 = go.Figure()

# IQR band
fig5.add_trace(go.Scatter(
    x=list(era_trend['season']) + list(era_trend['season'])[::-1],
    y=list(era_trend['p75_delta']) + list(era_trend['p25_delta'])[::-1],
    fill='toself',
    fillcolor='rgba(70,130,180,0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='25th–75th percentile',
    hoverinfo='skip',
))

# Median line
fig5.add_trace(go.Scatter(
    x=era_trend['season'],
    y=era_trend['med_delta'],
    mode='lines+markers',
    line=dict(color='steelblue', width=2.5),
    marker=dict(size=5),
    name='Median Δ/game',
    hovertemplate=(
        '<b>%{x}</b><br>Median Δ/game: %{y:.1f}<br>'
        'N backs: ' + era_trend['n_backs'].astype(str) +
        '<extra></extra>'
    ),
))

# Mean line
fig5.add_trace(go.Scatter(
    x=era_trend['season'],
    y=era_trend['mean_delta'],
    mode='lines',
    line=dict(color='navy', width=1.5, dash='dot'),
    name='Mean Δ/game',
    hoverinfo='skip',
))

fig5.add_hline(y=0, line_color='black', line_width=1.5)

# Era markers
for yr, lbl in [(1978, 'Rule changes\n(1978)'), (2000, '2000s passing era')]:
    fig5.add_vline(x=yr, line_dash='dot', line_color='rgba(150,0,0,0.4)', line_width=1.5,
                   annotation_text=lbl, annotation_position='top right',
                   annotation_font_size=9)

fig5.update_layout(
    title=dict(text='RB Carry-Adjusted Δ/Game by Season — All Featured Backs 1960–2024<br>'
                    '<sup>Median and mean delta for all backs with ≥400 rush yds and ≥8 games. Band = IQR.</sup>',
               x=0.5),
    xaxis=dict(title='Season'),
    yaxis=dict(title='Δ Rush Yards per Game (Carry-Adjusted)'),
    height=480,
    hovermode='x unified',
    legend=dict(orientation='h', y=-0.15),
)
fig5.show()

---
# Chart 6 — Named Backs: All Seasons Plotted

Every featured season for each GOAT-tier RB. X-axis = season, Y-axis = carry-adjusted Δ/game.
Shows career arcs, peak years, and decline — similar to the QB chart in the QB analysis.

In [11]:
fig6 = go.Figure()

# Background: all featured backs
fig6.add_trace(go.Scatter(
    x=rb_featured['season'],
    y=rb_featured['delta_rush_yds_pg'],
    mode='markers',
    marker=dict(color='rgba(200,200,200,0.18)', size=3.5),
    name='All featured backs',
    hoverinfo='skip',
    showlegend=False,
))

# League avg by season
fig6.add_trace(go.Scatter(
    x=era_trend['season'],
    y=era_trend['med_delta'],
    mode='lines',
    line=dict(color='rgba(80,80,80,0.45)', width=2, dash='dot'),
    name='League median',
    hoverinfo='skip',
))

for rb_name, color in NAMED_RBS.items():
    parts = rb_name.split()
    sub = rb_featured[
        rb_featured['player_name'].str.contains(parts[0], na=False) &
        rb_featured['player_name'].str.contains(parts[-1], na=False)
    ].sort_values('season')
    if sub.empty:
        continue
    hover = (
        '<b>' + sub['player_name'] + '</b> — ' + sub['team_abbrev'] + ' ' + sub['season'].astype(str) +
        '<br>Rush yds: ' + sub['total_rush_yds'].astype(int).astype(str) +
        '<br>Δ/game: <b>' + sub['delta_rush_yds_pg'].round(1).astype(str) + '</b>' +
        '<br>Δ season total: ' + sub['delta_rush_yds_total'].round(1).astype(str)
    )
    fig6.add_trace(go.Scatter(
        x=sub['season'],
        y=sub['delta_rush_yds_pg'],
        mode='markers+lines',
        marker=dict(color=color, size=8, opacity=0.88,
                    line=dict(width=0.8, color='white')),
        line=dict(color=color, width=1.5, dash='dot'),
        name=rb_name,
        hovertext=hover, hoverinfo='text',
    ))

fig6.add_hline(y=0, line_color='black', line_width=1.5)
for yr, lbl in [(1978, '1978 rule changes'), (2000, '2000 era')]:
    fig6.add_vline(x=yr, line_dash='dot', line_color='rgba(150,0,0,0.35)', line_width=1.5,
                   annotation_text=lbl, annotation_position='top right',
                   annotation_font_size=9)

fig6.update_layout(
    title=dict(text='All Featured Seasons — Named RBs: Carry-Adjusted Δ/Game vs. Season<br>'
                    '<sup>Each dot = one featured-back season. Dotted gray = league median.</sup>',
               x=0.5),
    xaxis=dict(title='Season'),
    yaxis=dict(title='Δ Rush Yards per Game (Carry-Adjusted)'),
    height=600,
    hovermode='closest',
    legend=dict(orientation='v', x=1.01, y=1,
                bgcolor='rgba(255,255,255,0.85)',
                bordercolor='lightgray', borderwidth=1),
)
fig6.show()

---
# Chart 7 — Rushing + Receiving Combined Delta

Some backs are more valuable as pass-catchers than as runners (e.g., Marshall Faulk, Roger Craig,
Christian McCaffrey). This chart adds the receiving delta to get a **total offensive impact** score.

Note: receiving delta uses the defense's average RB receiving yards allowed per game (not carry-adjusted,
since we don't have target splits). It's a noisier number but directionally correct.

In [12]:
rb_featured['delta_total_pg'] = rb_featured['delta_rush_yds_pg'] + rb_featured['delta_rec_yds_pg'].fillna(0)

top50_combined = rb_featured.nlargest(50, 'delta_total_pg').reset_index(drop=True)
top50_combined['label'] = top50_combined['player_name'].str.split().str[-1] + "'" + top50_combined['season'].astype(str).str[-2:]
top50_combined['color'] = top50_combined['season'].apply(era_color_game)

top50_combined['hover_comb'] = (
    '<b>' + top50_combined['player_name'] + '</b> — ' + top50_combined['team_abbrev'] + ' ' + top50_combined['season'].astype(str) +
    '<br>Rush Δ/game: ' + top50_combined['delta_rush_yds_pg'].round(1).astype(str) +
    '<br>Rec Δ/game:  ' + top50_combined['delta_rec_yds_pg'].fillna(0).round(1).astype(str) +
    '<br>Combined Δ/game: <b>' + top50_combined['delta_total_pg'].round(1).astype(str) + '</b>' +
    '<br>Rush yds: ' + top50_combined['total_rush_yds'].astype(int).astype(str) +
    '  Rec yds: ' + top50_combined['total_rec_yds'].fillna(0).astype(int).astype(str)
)

fig7 = go.Figure()
# Rush component
fig7.add_trace(go.Bar(
    x=top50_combined['label'],
    y=top50_combined['delta_rush_yds_pg'],
    name='Rush Δ/game',
    marker_color='steelblue',
    hovertext=top50_combined['hover_comb'],
    hoverinfo='text',
))
# Receiving component
fig7.add_trace(go.Bar(
    x=top50_combined['label'],
    y=top50_combined['delta_rec_yds_pg'].fillna(0),
    name='Rec Δ/game',
    marker_color='tomato',
    hoverinfo='skip',
))

fig7.add_hline(y=0, line_color='black', line_width=1.5)

fig7.update_layout(
    title=dict(text='Top 50 Seasons: Combined Rush + Receiving Δ per Game<br>'
                    '<sup>Blue = rush Δ/game (carry-adjusted). Red = receiving Δ/game (vs. defense RB rec avg). Stacked.</sup>',
               x=0.5),
    xaxis=dict(title='', tickangle=55, tickfont=dict(size=9)),
    yaxis=dict(title='Δ Yards per Game (Rush + Receiving)'),
    barmode='relative',
    height=520,
    hovermode='closest',
    legend=dict(orientation='h', y=-0.2),
)
fig7.show()

---
# Full Career Rankings Table

In [13]:
rb_career_q2 = rb_career_q[[
    'player_name', 'seasons', 'first_year', 'last_year',
    'avg_rush_yds', 'avg_delta_pg', 'avg_delta_total', 'career_delta',
    'avg_rush_td', 'avg_rec_yds', 'avg_delta_rec_pg'
]].sort_values('avg_delta_pg', ascending=False).reset_index(drop=True)

row_colors = []
for i, row in rb_career_q2.iterrows():
    if is_named(row['player_name']):
        row_colors.append('rgba(255,240,150,0.6)')
    else:
        row_colors.append('white' if i % 2 == 0 else '#f5f5f5')

fig_tbl = go.Figure(data=[go.Table(
    columnwidth=[160, 60, 50, 50, 80, 80, 90, 90, 70, 80, 90],
    header=dict(
        values=['<b>Player</b>', '<b>Seasons</b>', '<b>From</b>', '<b>To</b>',
                '<b>Avg Rush Yds</b>', '<b>Avg Δ/game</b>',
                '<b>Avg Δ/season</b>', '<b>Career Δ</b>',
                '<b>Avg Rush TDs</b>', '<b>Avg Rec Yds</b>', '<b>Avg Rec Δ/game</b>'],
        fill_color='#1a5276',
        font=dict(color='white', size=12),
        align='center', height=28,
    ),
    cells=dict(
        values=[
            rb_career_q2['player_name'],
            rb_career_q2['seasons'],
            rb_career_q2['first_year'],
            rb_career_q2['last_year'],
            rb_career_q2['avg_rush_yds'].round(0).astype(int),
            rb_career_q2['avg_delta_pg'].round(2),
            rb_career_q2['avg_delta_total'].round(1),
            rb_career_q2['career_delta'].round(1),
            rb_career_q2['avg_rush_td'].round(1),
            rb_career_q2['avg_rec_yds'].round(0).astype(int),
            rb_career_q2['avg_delta_rec_pg'].round(2),
        ],
        fill_color=[row_colors] * 11,
        font=dict(size=11),
        align=['left'] + ['center'] * 10,
        height=22,
    )
)])

fig_tbl.update_layout(
    title=dict(text=f'Full Career Rankings — {len(rb_career_q2)} RBs with 4+ Featured Seasons<br>'
                    '<sup>Yellow rows = named GOAT-tier RBs. Sorted by Avg Δ/game.</sup>',
               x=0.5),
    height=min(1600, 60 + 22 * len(rb_career_q2)),
    margin=dict(t=70, b=20, l=20, r=20),
)
fig_tbl.show()